# Training & Evaluation — KNN vs Logistic Regression
Kelompok 9 | COMP6577001 Machine Learning

**Fase 4:** Split → Train → Evaluate → Benchmark → Save models

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import joblib
import json
import time
from pathlib import Path
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)
import matplotlib.pyplot as plt
import seaborn as sns

PROCESSED_DIR = Path('../data/processed')
MODELS_DIR = Path('../models')
FIGURES_DIR = Path('../figures')
SPLIT_CONFIG = Path('../data/split/split_config.json')

MODELS_DIR.mkdir(exist_ok=True)
FIGURES_DIR.mkdir(exist_ok=True)
CLASS_NAMES = ['Fresh', 'Rotten']

## 4.1 Load & Split Data

In [ ]:
X = np.load(PROCESSED_DIR / 'X.npy')
y = np.load(PROCESSED_DIR / 'y.npy')

with open(SPLIT_CONFIG) as f:
    cfg = json.load(f)

print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')
print(f'Class distribution — Fresh: {(y==0).sum()}, Rotten: {(y==1).sum()}')
print(f'Split config: {cfg}')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=cfg['test_size'],
    random_state=cfg['random_state'],
    stratify=y
)
print(f'Train size: {len(X_train)}')
print(f'Test size:  {len(X_test)}')
print(f'Train class dist — Fresh: {(y_train==0).sum()}, Rotten: {(y_train==1).sum()}')
print(f'Test  class dist — Fresh: {(y_test==0).sum()},  Rotten: {(y_test==1).sum()}')

In [ ]:
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)
print('Features scaled (StandardScaler).')

In [ ]:

from sklearn.decomposition import PCA

pca = PCA(n_components=100, random_state=cfg['random_state'])
X_train_pca = pca.fit_transform(X_train_sc)
X_test_pca  = pca.transform(X_test_sc)

cumvar = np.cumsum(pca.explained_variance_ratio_)
print(f'Variance explained (100 components): {cumvar[-1]*100:.1f}%')
print(f'X_train_pca shape: {X_train_pca.shape}')

# PCA Cumulative Variance Curve
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 101), cumvar * 100, linewidth=2, color='steelblue')
ax.axhline(cumvar[-1] * 100, color='red', linestyle='--', alpha=0.7,
           label=f'{cumvar[-1]*100:.1f}% at 100 components')
ax.axhline(90, color='orange', linestyle=':', alpha=0.8, label='90% threshold')
ax.fill_between(range(1, 101), cumvar * 100, alpha=0.1, color='steelblue')
ax.set_xlabel('Number of Components')
ax.set_ylabel('Cumulative Variance Explained (%)')
ax.set_title('PCA — Cumulative Variance Explained')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pca_variance_curve.png', dpi=150)
plt.show()
print('Saved: pca_variance_curve.png')


## 4.2 Baseline: Logistic Regression

In [ ]:

logreg = LogisticRegression(max_iter=1000, random_state=cfg['random_state'])
logreg.fit(X_train_pca, y_train)
y_pred_lr = logreg.predict(X_test_pca)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=CLASS_NAMES, zero_division=0))

lr_metrics = {
    'accuracy':  accuracy_score(y_test, y_pred_lr),
    'precision': precision_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'recall':    recall_score(y_test, y_pred_lr, average='weighted', zero_division=0),
    'f1':        f1_score(y_test, y_pred_lr, average='weighted', zero_division=0),
}


In [ ]:
cm_lr = confusion_matrix(y_test, y_pred_lr)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix — Logistic Regression')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_logreg.png', dpi=150)
plt.show()

## 4.3 KNN with GridSearchCV

In [ ]:

param_grid = {'n_neighbors': [3, 5, 7, 9, 11]}
knn_cv = GridSearchCV(
    KNeighborsClassifier(),
    param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
knn_cv.fit(X_train_pca, y_train)

best_k = knn_cv.best_params_['n_neighbors']
print(f'Best k: {best_k}')
print(f'CV scores: {knn_cv.cv_results_["mean_test_score"]}')


In [ ]:
# Plot CV scores
k_values = param_grid['n_neighbors']
cv_scores = knn_cv.cv_results_['mean_test_score']

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(k_values, cv_scores, marker='o', linewidth=2)
ax.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax.set_xlabel('k (n_neighbors)')
ax.set_ylabel('CV F1 Score (weighted)')
ax.set_title('KNN GridSearchCV — k vs F1 Score')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'knn_cv_scores.png', dpi=150)
plt.show()

## 4.4 Evaluate KNN

In [ ]:

knn_best = knn_cv.best_estimator_
y_pred_knn = knn_best.predict(X_test_pca)

print(f'=== KNN (k={best_k}) ===')
print(classification_report(y_test, y_pred_knn, target_names=CLASS_NAMES, zero_division=0))

knn_metrics = {
    'accuracy':  accuracy_score(y_test, y_pred_knn),
    'precision': precision_score(y_test, y_pred_knn, average='weighted', zero_division=0),
    'recall':    recall_score(y_test, y_pred_knn, average='weighted', zero_division=0),
    'f1':        f1_score(y_test, y_pred_knn, average='weighted', zero_division=0),
}


In [ ]:
cm_knn = confusion_matrix(y_test, y_pred_knn)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_knn, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title(f'Confusion Matrix — KNN (k={best_k})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_knn.png', dpi=150)
plt.show()

In [ ]:
# Side-by-side confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cm, title, cmap in [
    (axes[0], cm_lr, 'Logistic Regression', 'Blues'),
    (axes[1], cm_knn, f'KNN (k={best_k})', 'Greens'),
]:
    sns.heatmap(cm, annot=True, fmt='d', cmap=cmap,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')
    ax.set_title(title)
plt.suptitle('Confusion Matrix Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'confusion_matrix_comparison.png', dpi=150)
plt.show()

## 4.5 Latency Benchmark

In [ ]:

N_RUNS = 100
feat_extraction_ms = 7.16  # measured in notebook 02

def benchmark_full(model, n=N_RUNS):
    times = []
    for _ in range(n):
        t     = time.perf_counter()
        x_sc  = scaler.transform(X_test[:1])
        x_pca = pca.transform(x_sc)
        model.predict(x_pca)
        times.append((time.perf_counter() - t) * 1000)
    return np.mean(times), np.std(times)

lr_lat_mean,  lr_lat_std  = benchmark_full(logreg)
knn_lat_mean, knn_lat_std = benchmark_full(knn_best)

print(f'LogReg inference (scaler+PCA+model): {lr_lat_mean:.2f} ± {lr_lat_std:.2f} ms')
print(f'KNN    inference (scaler+PCA+model): {knn_lat_mean:.2f} ± {knn_lat_std:.2f} ms')
print(f'Feature extraction (pre-measured):   {feat_extraction_ms:.2f} ms')
print(f'Total pipeline LogReg: {feat_extraction_ms + lr_lat_mean:.2f} ms')
print(f'Total pipeline KNN:    {feat_extraction_ms + knn_lat_mean:.2f} ms')
print(f'LogReg PASS (<100ms): {(feat_extraction_ms + lr_lat_mean) < 100}')
print(f'KNN    PASS (<100ms): {(feat_extraction_ms + knn_lat_mean) < 100}')

# Latency bar chart
bar_labels = ['Feature\nExtraction', 'Inference\n(LogReg)', 'Inference\n(KNN k=3)',
              'Total\n(LogReg)', 'Total\n(KNN k=3)']
bar_values = [feat_extraction_ms, lr_lat_mean, knn_lat_mean,
              feat_extraction_ms + lr_lat_mean, feat_extraction_ms + knn_lat_mean]
bar_colors = ['steelblue', 'lightsteelblue', 'seagreen', 'lightsteelblue', 'seagreen']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(bar_labels, bar_values, color=bar_colors, edgecolor='white')
ax.axhline(100, color='red', linestyle='--', linewidth=1.5, label='Requirement: 100 ms')
ax.set_ylabel('Latency (ms)')
ax.set_title('Pipeline Latency Breakdown')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, bar_values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.3,
            f'{val:.1f}ms', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'latency_breakdown.png', dpi=150)
plt.show()
print('Saved: latency_breakdown.png')


## 4.6 Summary Table

In [ ]:
import pandas as pd

summary = pd.DataFrame([
    {
        'Model': 'Logistic Regression (baseline)',
        'Accuracy': lr_metrics['accuracy'],
        'Precision': lr_metrics['precision'],
        'Recall': lr_metrics['recall'],
        'F1 Score': lr_metrics['f1'],
        'Inference (ms)': lr_lat_mean,
        'Total pipeline (ms)': feat_extraction_ms + lr_lat_mean,
    },
    {
        'Model': f'KNN (k={best_k})',
        'Accuracy': knn_metrics['accuracy'],
        'Precision': knn_metrics['precision'],
        'Recall': knn_metrics['recall'],
        'F1 Score': knn_metrics['f1'],
        'Inference (ms)': knn_lat_mean,
        'Total pipeline (ms)': feat_extraction_ms + knn_lat_mean,
    },
])

float_cols = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'Inference (ms)', 'Total pipeline (ms)']
summary[float_cols] = summary[float_cols].round(4)
print(summary.to_string(index=False))

In [ ]:
# Bar chart comparison
metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
x = np.arange(len(metrics_to_plot))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 5))
bars1 = ax.bar(x - width/2, summary.loc[0, metrics_to_plot], width, label='Logistic Regression', color='steelblue')
bars2 = ax.bar(x + width/2, summary.loc[1, metrics_to_plot], width, label=f'KNN (k={best_k})', color='seagreen')

ax.set_xticks(x)
ax.set_xticklabels(metrics_to_plot)
ax.set_ylim(0, 1.1)
ax.set_ylabel('Score')
ax.set_title('Model Comparison — KNN vs Logistic Regression')
ax.legend()
ax.axhline(0.8, color='red', linestyle='--', alpha=0.6, label='Target (0.80)')

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'model_comparison.png', dpi=150)
plt.show()

## 4.6 Save Models

In [ ]:

joblib.dump(knn_best, MODELS_DIR / 'knn_model.pkl')
joblib.dump(logreg,   MODELS_DIR / 'baseline_logreg.pkl')
joblib.dump(scaler,   MODELS_DIR / 'scaler.pkl')
joblib.dump(pca,      MODELS_DIR / 'pca.pkl')
print('Saved:')
print(' - models/knn_model.pkl')
print(' - models/baseline_logreg.pkl')
print(' - models/scaler.pkl')
print(' - models/pca.pkl')
